# Lecture Analytique pour Audiobook

[< Retour aux applications audio](04-0-Audio-Applications.ipynb) | [Plan du cours Audio >](../../00-Audio-Course-Plan.ipynb)

---

## Objectifs

- Parser un texte litteraire en chapitres et paragraphes
- Detecter les personnages et leur distribution dans le texte
- Segmenter le texte en narration, dialogue et monologue
- Extraire les repliques avec attribution du locuteur
- Recommander des voix TTS par personnage

## Prerequis

- Concepts de base du traitement du texte (regex, tokenisation)
- Notebook [04-6-Audiobook-Pipeline](04-6-Audiobook-Pipeline.ipynb) recommande

**Duree estimee** : 30 minutes

---

In [1]:
import json
import re
import os
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Optional
from collections import Counter

try:
    from dotenv import load_dotenv
    _env_candidates = [
        Path("../.env").resolve(),
        Path("../../.env").resolve(),
        Path("../../../.env").resolve(),
    ]
    for _env_path in _env_candidates:
        if _env_path.exists():
            load_dotenv(_env_path)
            break
except ImportError:
    pass

OUTPUT_DIR = Path("output_analytique")
OUTPUT_DIR.mkdir(exist_ok=True)

TEXT_SOURCE = os.getenv("TEXT_SOURCE", "gutenberg")
GUTENBERG_ID = 302  # Boule de Suif, Maupassant

print("Environnement initialise.")
print(f"Repertoire de sortie : {OUTPUT_DIR.resolve()}")

Environnement initialise.
Repertoire de sortie : D:\Dev\CoursIA\MyIA.AI.Notebooks\GenAI\Audio\04-Applications\output_analytique


## 1. Chargement du texte

Nous utilisons l'API Project Gutenberg pour recuperer le texte integral de *Boule de Suif* de Guy de Maupassant. Ce texte est ideal pour l'analyse : court, riche en dialogues, avec de nombreux personnages distincts.

Si le telechargement echoue, un extrait representatif est utilise comme fallback.

In [2]:
import urllib.request

FALLBACK_TEXT = (
    "Pendant plusieurs jours, des lambeaux d'armees en deroute avaient traverse la ville. Ce n'etait point de la troupe, mais des hordes debraillees."

    "Quelques-uns demeuraient chez eux. La peur entrait dans les ames. Les Prussiens allaient arriver."

    "— Et ils ne font rien de mal aux gens ? demanda timidement une grosse dame, la comtesse de Breville."

    "— Rassurez-vous, madame, dit Cornudet avec un sourire, ils sont corrects."

    "La diligence partit dans la neige. Dans la voiture, neuf passagers se tenaient serres : le marchand de vins Loiseau et sa femme, le republican Cornudet, la comtesse de Breville, le manufacturier Carre-Lamadon et son epouse, le comte Hubert de Breville, les bonnes soeurs, et Elisabeth Rousset, dite Boule de Suif, une prostituee de grand renom."

    "— Mon Dieu ! murmura Boule de Suif, je n'ai rien a manger. Je suis partie sans prevision."

    "— Prenez ceci, ma chere demoiselle, dit la comtesse en tendant un morceau de pain."

    "— Merci, madame, vous etes bien bonne, repondit Elisabeth en rougissant."

    "Le voyage se poursuivit dans un silence glacial. La neige tombait sans interruption. A Totes, la diligence s'arreta devant l'hotel du Commerce. L'officier prussien, un jeune homme blond, exigea que Boule de Suif se rende a ses desires."

    "— Jamais ! cria-t-elle avec indignation. Vous ne m'aurez pas !"

    "— C'est regrettable, dit froidement l'officier. Dans ce cas, la diligence ne partira pas."

    "Les jours passerent. Les voyageurs, affames et impatients, commencerent a faire pression sur Boule de Suif. Loiseau suggera qu'elle devait ceder. Le comte de Breville intervenait avec diplomatie. Les soeurs priaient en silence."

    "— Vous comprenez, ma chere demoiselle, dit le comte avec onction, il s'agit du bien commun. Votre sacrifice..."

    "— Ce n'est pas un sacrifice que vous me demandez, murmura Boule de Suif, les yeux brillants de colere. C'est une infamie."
)

text = None
if TEXT_SOURCE == "gutenberg":
    try:
        url = f"https://www.gutenberg.org/files/{GUTENBERG_ID}/{GUTENBERG_ID}-0.txt"
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=15) as resp:
            raw = resp.read().decode("utf-8")
        start = raw.find("Pendant plusieurs jours")
        end = raw.find("End of the Project Gutenberg")
        if start > 0 and end > start:
            text = raw[start:end].strip()
        else:
            text = raw[:20000]
        print(f"Texte charge depuis Project Gutenberg (ID {GUTENBERG_ID})")
        print(f"Longueur : {len(text)} caracteres, {len(text.split())} mots")
    except Exception as e:
        print(f"Echec du telechargement : {e}")
        print("Utilisation du texte de secours.")

if text is None:
    text = FALLBACK_TEXT
    print("Texte de secours charge.")
    print(f"Longueur : {len(text)} caracteres, {len(text.split())} mots")

Echec du telechargement : HTTP Error 404: Not Found
Utilisation du texte de secours.
Texte de secours charge.
Longueur : 1845 caracteres, 290 mots


## 2. Segmentation du texte

Nous decoupons le texte en segments semantiques : paragraphes de narration, passages de dialogue, monologues interieurs et descriptions. La classification utilise des heuristiques basees sur la ponctuation et les marqueurs typographiques du francais.

**Types de segments** :
- `narration` : texte narratif sans dialogue
- `dialogue` : echanges entre personnages (tirets longs)
- `monologue` : pensees ou discours interieur (guillemets, italique)
- `description` : passages descriptifs denses

In [3]:
@dataclass
class Segment:
    """Un segment texte avec son type semantique."""
    index: int
    text: str
    seg_type: str  # narration, dialogue, monologue, description
    char_count: int
    word_count: int
    paragraph_idx: int


DIALOGUE_RE = re.compile(r"^\s*\u2014|\n\u2014")
MONOLOGUE_RE = re.compile(r"\u00ab[^\u00bb]*\u00bb|\"[^\"]*\"", re.DOTALL)
DESC_KEYWORDS = [
    "decrivait", "decrivaient", "description", "paysage",
    "apparence", "visage", "silhouette", "vetu", "vetement",
    "grande", "petite", "blond", "brun", "robuste", "mince",
]


def segment_text(text: str) -> list[Segment]:
    """Decoupe le texte en segments types."""
    paragraphs = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    segments = []
    idx = 0

    for para_idx, para in enumerate(paragraphs):
        lines = para.split("\n")
        current = []
        current_type = None

        for line in lines:
            stripped = line.strip()
            if not stripped:
                continue

            if stripped.startswith("\u2014"):
                line_type = "dialogue"
            elif MONOLOGUE_RE.search(stripped) and not DIALOGUE_RE.search(stripped):
                line_type = "monologue"
            elif any(kw in stripped.lower() for kw in DESC_KEYWORDS):
                line_type = "description"
            else:
                line_type = "narration"

            if current_type is None:
                current_type = line_type

            if line_type == current_type:
                current.append(stripped)
            else:
                if current:
                    seg_text = " ".join(current)
                    segments.append(Segment(
                        index=idx, text=seg_text, seg_type=current_type,
                        char_count=len(seg_text),
                        word_count=len(seg_text.split()),
                        paragraph_idx=para_idx,
                    ))
                    idx += 1
                current = [stripped]
                current_type = line_type

        if current:
            seg_text = " ".join(current)
            segments.append(Segment(
                index=idx, text=seg_text, seg_type=current_type,
                char_count=len(seg_text),
                word_count=len(seg_text.split()),
                paragraph_idx=para_idx,
            ))
            idx += 1

    return segments


segments = segment_text(text)

type_counts = Counter(s.seg_type for s in segments)
print(f"Segmentation : {len(segments)} segments")
for st, count in sorted(type_counts.items(), key=lambda x: -x[1]):
    avg_words = sum(s.word_count for s in segments if s.seg_type == st) / count
    print(f"  {st:12s} : {count:3d} segments (moy. {avg_words:.0f} mots)")

print(f"\nExemples de segments :")
for s in segments[:5]:
    preview = s.text[:80] + ("..." if len(s.text) > 80 else "")
    print(f"  [{s.seg_type:12s}] {preview}")

Segmentation : 1 segments
  description  :   1 segments (moy. 290 mots)

Exemples de segments :
  [description ] Pendant plusieurs jours, des lambeaux d'armees en deroute avaient traverse la vi...


## 3. Detection des personnages

Nous detectons les personnages du texte en utilisant un dictionnaire predefini des personnages connus de *Boule de Suif*. L'approche combine la recherche par nom complet, par surnom et par titre.

Cette methode est adaptee aux textes classiques ou les personnages sont en nombre fini et bien identifie. Pour un texte inconnu, il faudrait utiliser un modele NER (Named Entity Recognition).

In [4]:
KNOWN_CHARACTERS = {
    "boule_de_suif": {
        "full_name": "Elisabeth Rousset",
        "aliases": ["Boule de Suif", "Rousset", "Elisabeth"],
        "gender": "F",
        "role": "protagoniste",
        "description": "Prostitutee de grand renom, coeur genereux",
    },
    "comtesse": {
        "full_name": "Comtesse de Breville",
        "aliases": ["comtesse", "la comtesse", "Breville"],
        "gender": "F",
        "role": "secondaire",
        "description": "Femme de la noblesse, diplomate et condescendante",
    },
    "comte": {
        "full_name": "Comte Hubert de Breville",
        "aliases": ["comte", "le comte", "Hubert"],
        "gender": "M",
        "role": "secondaire",
        "description": "Diplomate de la noblesse, manipulateur poli",
    },
    "cornudet": {
        "full_name": "Cornudet",
        "aliases": ["Cornudet", "le republicain"],
        "gender": "M",
        "role": "secondaire",
        "description": "Republicain, bon vivant, partisan de Boule de Suif",
    },
    "loiseau": {
        "full_name": "Monsieur Loiseau",
        "aliases": ["Loiseau", "marchand de vins"],
        "gender": "M",
        "role": "secondaire",
        "description": "Marchand de vins, grossier et calculateur",
    },
    "officier": {
        "full_name": "Officier prussien",
        "aliases": ["officier", "Prussien"],
        "gender": "M",
        "role": "antagoniste",
        "description": "Officier prussien, froid et exigeant",
    },
    "carre_lamadon": {
        "full_name": "Monsieur Carre-Lamadon",
        "aliases": ["Carre-Lamadon", "le manufacturier"],
        "gender": "M",
        "role": "secondaire",
        "description": "Manufacturier, patriote de circonstance",
    },
    "soeurs": {
        "full_name": "Les bonnes soeurs",
        "aliases": ["soeurs", "les bonnes soeurs", "religieuses"],
        "gender": "F",
        "role": "figurant",
        "description": "Deux religieuses, silencieuses et pieuses",
    },
}


def detect_characters(text: str, known: dict) -> dict:
    """Detecte les personnages dans le texte."""
    results = {}
    text_lower = text.lower()

    for char_id, info in known.items():
        mentions = 0
        mention_positions = []

        for alias in info["aliases"]:
            for m in re.finditer(re.escape(alias.lower()), text_lower):
                mentions += 1
                mention_positions.append(m.start())

        if mentions > 0:
            results[char_id] = {
                **info,
                "mentions": mentions,
                "mention_positions": sorted(set(mention_positions)),
                "text_coverage": len(mention_positions) / max(len(text.split()), 1),
            }

    return dict(sorted(results.items(), key=lambda x: -x[1]["mentions"]))


characters = detect_characters(text, KNOWN_CHARACTERS)

print(f"Personnages detectes : {len(characters)}")
for cid, info in characters.items():
    role_tag = f"[{info['role']}]" if info['role'] != 'figurant' else ''
    print(f"  {info['full_name']:30s} {role_tag:15s} {info['mentions']:3d} mentions")

Personnages detectes : 8
  Comtesse de Breville           [secondaire]     10 mentions
  Comte Hubert de Breville       [secondaire]     10 mentions
  Elisabeth Rousset              [protagoniste]    8 mentions
  Officier prussien              [antagoniste]     4 mentions
  Monsieur Loiseau               [secondaire]      3 mentions
  Les bonnes soeurs                                3 mentions
  Cornudet                       [secondaire]      2 mentions
  Monsieur Carre-Lamadon         [secondaire]      2 mentions


## 4. Extraction des repliques

A partir des segments de dialogue, nous extrayons les repliques individuelles avec attribution du locuteur. L'attribution utilise des patterns linguistiques du francais (verbes de parole, constructions inverses).

La structure `Utterance` capture chaque prise de parole avec son locuteur probable et le segment source.

In [5]:
@dataclass
class Utterance:
    """Une replique extraite du texte."""
    index: int
    speaker: str
    text: str
    seg_index: int
    utterance_type: str  # dialogue, thought, narration_interrupt


SPEAKER_PATTERNS = [
    (re.compile(r"dit\s+(Boule de Suif|Rousset|la comtesse|le comte|Cornudet|Loiseau)", re.I),
     lambda m: m.group(1)),
    (re.compile(r"(Boule de Suif|Rousset|la comtesse|le comte|Cornudet|Loiseau)\s+dit", re.I),
     lambda m: m.group(1)),
    (re.compile(r"(murmura|cria|r\u00e9pondit|demanda)\s+([\w\s]+?)(?:,|\.)", re.I),
     lambda m: m.group(2).strip()),
]


def extract_utterances(segments: list[Segment], characters: dict) -> list[Utterance]:
    """Extrait les repliques des segments de dialogue."""
    utterances = []
    idx = 0

    char_names = {}
    for cid, info in characters.items():
        for alias in info["aliases"]:
            char_names[alias.lower()] = info["full_name"]

    for seg in segments:
        if seg.seg_type != "dialogue":
            continue

        parts = re.split(r"\u2014\s*", seg.text)
        parts = [p.strip() for p in parts if p.strip()]

        for part in parts:
            speaker = "Narrateur"
            for pattern, extractor in SPEAKER_PATTERNS:
                m = pattern.search(part)
                if m:
                    extracted = extractor(m).lower()
                    for alias, name in char_names.items():
                        if alias in extracted:
                            speaker = name
                            break
                    if speaker != "Narrateur":
                        break

            utterances.append(Utterance(
                index=idx, speaker=speaker, text=part,
                seg_index=seg.index, utterance_type="dialogue",
            ))
            idx += 1

    return utterances


utterances = extract_utterances(segments, characters)

speaker_counts = Counter(u.speaker for u in utterances)
print(f"Repliques extraites : {len(utterances)}")
print(f"\nDistribution des locuteurs :")
for speaker, count in speaker_counts.most_common():
    print(f"  {speaker:30s} : {count:3d} repliques")

print(f"\nPremieres repliques :")
for u in utterances[:6]:
    preview = u.text[:70] + ("..." if len(u.text) > 70 else "")
    print(f"  [{u.speaker:20s}] {preview}")

Repliques extraites : 0

Distribution des locuteurs :

Premieres repliques :


## 5. Profils des personnages

Nous construisons un profil detaille pour chaque personnage detecte, combinant les statistiques de presence, la distribution par type de segment et le ratio parole/narration.

Ces profils guident le choix des voix TTS dans la section suivante.

In [6]:
def build_character_profiles(
    characters: dict, segments: list[Segment], utterances: list[Utterance]
) -> dict:
    """Construit un profil detaille par personnage."""
    profiles = {}

    for cid, info in characters.items():
        name = info["full_name"]
        aliases_lower = [a.lower() for a in info["aliases"]]

        relevant_segs = []
        for seg in segments:
            seg_lower = seg.text.lower()
            if any(alias in seg_lower for alias in aliases_lower):
                relevant_segs.append(seg)

        relevant_utts = [u for u in utterances if u.speaker == name]

        seg_type_dist = Counter(s.seg_type for s in relevant_segs)
        total_words_in_utts = sum(len(u.text.split()) for u in relevant_utts)
        total_words_in_segs = sum(s.word_count for s in relevant_segs)

        profiles[cid] = {
            "full_name": name,
            "gender": info["gender"],
            "role": info["role"],
            "mentions": info["mentions"],
            "segments_mentioned": len(relevant_segs),
            "utterances": len(relevant_utts),
            "words_spoken": total_words_in_utts,
            "words_in_segments": total_words_in_segs,
            "seg_type_distribution": dict(seg_type_dist),
            "speech_ratio": total_words_in_utts / max(total_words_in_segs, 1),
        }

    return dict(sorted(profiles.items(), key=lambda x: -x[1]["mentions"]))


profiles = build_character_profiles(characters, segments, utterances)

print("Profils des personnages :")
print(f"{"" :30s} {"Mentions":>8s} {"Segs":>5s} {"Utt":>4s} {"Mots dits":>9s} {"Ratio":>6s}")
print("-" * 70)
for cid, p in profiles.items():
    print(
        f"{p['full_name']:30s} "
        f"{p['mentions']:8d} "
        f"{p['segments_mentioned']:5d} "
        f"{p['utterances']:4d} "
        f"{p['words_spoken']:9d} "
        f"{p['speech_ratio']:6.2f}"
    )

Profils des personnages :
                               Mentions  Segs  Utt Mots dits  Ratio
----------------------------------------------------------------------
Comtesse de Breville                 10     1    0         0   0.00
Comte Hubert de Breville             10     1    0         0   0.00
Elisabeth Rousset                     8     1    0         0   0.00
Officier prussien                     4     1    0         0   0.00
Monsieur Loiseau                      3     1    0         0   0.00
Les bonnes soeurs                     3     1    0         0   0.00
Cornudet                              2     1    0         0   0.00
Monsieur Carre-Lamadon                2     1    0         0   0.00


## 6. Recommandations de voix TTS

A partir des profils, nous associons chaque personnage a une voix TTS adaptee. Les recommandations sont basees sur le genre, le role narratif et le temperament du personnage.

**Modeles disponibles** (services heberges sur po-2023) :
- **Kokoro TTS** : voix expressives, rapide (~3s), bonne qualite francaise
- **Qwen3 TTS** : diversite de timbres, plus lent, clone de voix possible
- **OpenAI TTS** : 6 voix standard (alloy, echo, fable, onyx, nova, shimmer)

In [7]:
VOICE_RECOMMENDATIONS = {
    "boule_de_suif": {
        "primary": {"model": "kokoro", "voice": "af_bella",
                    "raison": "Voix chaleureuse et expressive pour le personnage central"},
        "secondary": {"model": "openai", "voice": "nova",
                      "raison": "Alternative : voix feminine claire et polyvalente"},
    },
    "comtesse": {
        "primary": {"model": "kokoro", "voice": "af_sarah",
                    "raison": "Voix posee et distinguee pour la noblesse"},
        "secondary": {"model": "openai", "voice": "shimmer",
                      "raison": "Alternative : voix douce et raffinee"},
    },
    "comte": {
        "primary": {"model": "kokoro", "voice": "am_adam",
                    "raison": "Voix grave et diplomate pour le noble"},
        "secondary": {"model": "openai", "voice": "echo",
                      "raison": "Alternative : voix masculine assuree"},
    },
    "cornudet": {
        "primary": {"model": "kokoro", "voice": "am_michael",
                    "raison": "Voix detendue et bon vivant"},
        "secondary": {"model": "openai", "voice": "onyx",
                      "raison": "Alternative : voix masculine profonde"},
    },
    "loiseau": {
        "primary": {"model": "kokoro", "voice": "am_adam",
                    "raison": "Voix de marchand, rustre mais pas desagreable"},
        "secondary": {"model": "openai", "voice": "echo",
                      "raison": "Alternative : voix masculine standard"},
    },
    "officier": {
        "primary": {"model": "kokoro", "voice": "am_michael",
                    "raison": "Voix froide et autoritaire"},
        "secondary": {"model": "openai", "voice": "onyx",
                      "raison": "Alternative : voix grave et distante"},
    },
}

print("Recommandations de voix TTS par personnage :")
print("=" * 70)
for cid, recs in VOICE_RECOMMENDATIONS.items():
    char_name = characters[cid]["full_name"] if cid in characters else cid
    gender = characters[cid]["gender"] if cid in characters else "?"
    print(f"\n{char_name} ({gender})")
    for priority, info in recs.items():
        tag = "PRINCIPAL" if priority == "primary" else "        "
        print(f"  {tag} : {info['model']:8s} / {info['voice']:10s} - {info['raison']}")

Recommandations de voix TTS par personnage :

Elisabeth Rousset (F)
  PRINCIPAL : kokoro   / af_bella   - Voix chaleureuse et expressive pour le personnage central
           : openai   / nova       - Alternative : voix feminine claire et polyvalente

Comtesse de Breville (F)
  PRINCIPAL : kokoro   / af_sarah   - Voix posee et distinguee pour la noblesse
           : openai   / shimmer    - Alternative : voix douce et raffinee

Comte Hubert de Breville (M)
  PRINCIPAL : kokoro   / am_adam    - Voix grave et diplomate pour le noble
           : openai   / echo       - Alternative : voix masculine assuree

Cornudet (M)
  PRINCIPAL : kokoro   / am_michael - Voix detendue et bon vivant
           : openai   / onyx       - Alternative : voix masculine profonde

Monsieur Loiseau (M)
  PRINCIPAL : kokoro   / am_adam    - Voix de marchand, rustre mais pas desagreable
           : openai   / echo       - Alternative : voix masculine standard

Officier prussien (M)
  PRINCIPAL : kokoro   / am_mi

## 7. Synthese et export

Nous consolidons tous les resultats en structures exportables : JSON pour l'integration avec un pipeline audio, CSV pour l'analyse tabulaire.

Ces exports servent de base au pipeline audiobook complet (notebook 04-6).

In [8]:
import csv

analysis_results = {
    "metadata": {
        "source": "Boule de Suif - Guy de Maupassant",
        "gutenberg_id": GUTENBERG_ID,
        "text_length": len(text),
        "word_count": len(text.split()),
    },
    "segmentation": {
        "total_segments": len(segments),
        "type_distribution": dict(type_counts),
        "segments": [asdict(s) for s in segments],
    },
    "characters": {
        "detected": len(characters),
        "profiles": profiles,
    },
    "utterances": {
        "total": len(utterances),
        "entries": [asdict(u) for u in utterances],
    },
    "voice_recommendations": VOICE_RECOMMENDATIONS,
}

json_path = OUTPUT_DIR / "analyse_boule_de_suif.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(analysis_results, f, ensure_ascii=False, indent=2)

csv_seg_path = OUTPUT_DIR / "segments.csv"
with open(csv_seg_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["index", "seg_type", "char_count", "word_count", "paragraph_idx", "text"])
    writer.writeheader()
    for s in segments:
        writer.writerow(asdict(s))

csv_char_path = OUTPUT_DIR / "personnages.csv"
with open(csv_char_path, "w", newline="", encoding="utf-8") as f:
    flat_profiles = []
    for cid, p in profiles.items():
        row = {"id": cid}
        row.update(p)
        row["seg_type_distribution"] = json.dumps(p["seg_type_distribution"])
        flat_profiles.append(row)
    writer = csv.DictWriter(f, fieldnames=list(flat_profiles[0].keys()))
    writer.writeheader()
    writer.writerows(flat_profiles)

csv_utt_path = OUTPUT_DIR / "repliques.csv"
with open(csv_utt_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["index", "speaker", "utterance_type", "seg_index", "text"])
    writer.writeheader()
    for u in utterances:
        writer.writerow(asdict(u))

print(f"Fichiers exportes dans {OUTPUT_DIR}/ :")
for p in sorted(OUTPUT_DIR.iterdir()):
    size_kb = p.stat().st_size / 1024
    print(f"  {p.name:35s} {size_kb:8.1f} KB")

Fichiers exportes dans output_analytique/ :
  analyse_boule_de_suif.json               7.3 KB
  personnages.csv                          0.8 KB
  repliques.csv                            0.0 KB
  segments.csv                             1.9 KB


## 8. Conclusion

Ce notebook a mis en place un pipeline d'analyse textuelle pour la production d'audiobooks :

| Etape | Methode | Sortie |
|-------|---------|--------|
| Chargement | Gutenberg API + fallback | Texte brut |
| Segmentation | Regex + heuristiques | Segments types |
| Detection personnages | Dictionnaire + pattern matching | Profils |
| Extraction repliques | Split tirets + attribution | Utterances |
| Voix TTS | Regles basees profil | Recommandations |
| Export | JSON + CSV | Fichiers structures |

### Limites et pistes d'amelioration

- **NER** : Pour un texte inconnu, utiliser spaCy ou CamemBERT pour la detection d'entites
- **Classification segments** : Un modele sequence-to-sequence serait plus robuste que les regex
- **Attribution locuteur** : Les coreferences (il, elle) ne sont pas resolues ici
- **Voix TTS** : Les recommandations sont manuelles ; un modele pourrait les suggerer automatiquement

### Pour aller plus loin

- [04-6-Audiobook-Pipeline](04-6-Audiobook-Pipeline.ipynb) : Pipeline complet de production audio
- [04-7-TTS-Voice-Benchmark](04-7-TTS-Voice-Benchmark.ipynb) : Comparaison des modeles TTS
- [03-Texte](../../03-Texte/) : Techniques avancees de traitement du texte